# Pig Posture Classification

**Automatically classify pig postures (Standing / Sitting / Lying) from OAK-D camera images.**

### How to use (3 steps):
1. **Click `Runtime > Run all`** at the top — this sets everything up
2. **Connect your Google Drive** when prompted (click the link, sign in, paste the code)
3. **Set your image folder path** in Step 2 below

That's it! The pipeline will crop images, run the AI model, and give you:
- A **heatmap** showing all 20 pigs' postures over time
- **Per-pig heatmaps** — individual timeline for each pig
- **Per-pig CSV files** — every prediction for each pig separately
- A **summary CSV** — posture breakdown per timestamp folder
- A **full predictions CSV** — every single frame prediction
- Everything zipped into **one download**

---

## Step 1: Setup (automatic)
This installs everything needed. Just run it and wait ~30 seconds.

In [ ]:
!pip install -q torch torchvision opencv-python-headless matplotlib numpy pandas
print("Setup complete!")

In [ ]:
# Connect to Google Drive so we can access your images
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive connected!")

## Step 2: Set your image folder path

Change the path below to where your OAK-D images are on Google Drive.

Your folder should contain timestamp subfolders like:
```
Pictures_OAk/
  20260211-09-17-49/
    pig0_ir_20260211-09-17-49.jpg
    pig0_depth_20260211-09-17-49.raw
    pig1_ir_20260211-09-17-49.jpg
    ...
  20260211-09-18-52/
    ...
```

In [ ]:
# ============================================================
#  CHANGE THIS PATH to your image folder on Google Drive
# ============================================================

DATA_DIR = "/content/drive/MyDrive/PAAL_data/Fed_pig/Pictures_OAk"

# ============================================================

import os
if not os.path.isdir(DATA_DIR):
    print(f"ERROR: Folder not found: {DATA_DIR}")
    print(f"")
    print(f"Go to your Google Drive in a browser, find the image folder,")
    print(f"right-click > 'Copy path' and paste it above.")
    print(f"")
    print(f"The path should start with /content/drive/MyDrive/...")
else:
    folders = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and d[0].isdigit()]
    print(f"Found {len(folders)} timestamp folders in your data.")
    if folders:
        print(f"First: {sorted(folders)[0]}")
        print(f"Last:  {sorted(folders)[-1]}")
    print(f"")
    print(f"Looks good! Proceed to Step 3.")

## Step 3: Run the pipeline

Just run the cells below. The model is built into this notebook — no extra files needed.

**Estimated time:** ~30 minutes for 3,000+ folders on Colab GPU.

In [ ]:
# ── All pipeline code (don't modify) ─────────────────────────

import csv
import re
import shutil
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import pandas as pd

OUTPUT_DIR = "/content/paal_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224
TOF_W, TOF_H = 640, 480
DEPTH_THRESHOLD = 1463
BAR_MARGIN = 50
CROP_TOF = (120, 30, 500, 480)
POSTURE3_CLASSES = {0: "standing", 1: "sitting", 2: "lying"}

FILENAME_RE = re.compile(
    r"^pig(\d+)_(depth_vis|depth|ir_vis|ir|rgb_aligned|rgb)_"
    r"(\d{8}-\d{2}-\d{2}-\d{2})(_cropped)?\.(jpg|raw)$"
)
FOLDER_RE = re.compile(r"^\d{8}-\d{2}-\d{2}-\d{2}$")


class SingleModalModel(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, pretrained=False, backbone="mobilenet_v2"):
        super().__init__()
        net = models.mobilenet_v2(weights=None)
        if in_channels != 3:
            old = net.features[0][0]
            net.features[0][0] = nn.Conv2d(in_channels, old.out_channels,
                kernel_size=old.kernel_size, stride=old.stride,
                padding=old.padding, bias=old.bias is not None)
        self.features = net.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(net.last_channel, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


def load_and_preprocess(path):
    if not path or not os.path.exists(path):
        return None
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0


def crop_box_for_size(w, h):
    x1, y1, x2, y2 = CROP_TOF
    return int(x1 * w / TOF_W), int(y1 * h / TOF_H), int(x2 * w / TOF_W), int(y2 * h / TOF_H)


def crop_all_images(data_dir):
    folders = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and d[0].isdigit())
    print(f"Cropping images in {len(folders)} folders...")
    total, skipped, corrupt = 0, 0, 0
    for fi, folder in enumerate(folders):
        folder_path = os.path.join(data_dir, folder)
        for fname in os.listdir(folder_path):
            if not fname.endswith(".jpg") or "_cropped" in fname or not re.match(r"^pig\d+_", fname):
                continue
            src = os.path.join(folder_path, fname)
            dst = os.path.splitext(src)[0] + "_cropped.jpg"
            if os.path.exists(dst):
                skipped += 1
                continue
            if os.path.getsize(src) < 1000:
                corrupt += 1
                continue
            img = cv2.imread(src)
            if img is None or img.shape[0] < 10 or img.shape[1] < 10:
                corrupt += 1
                continue
            h, w = img.shape[:2]
            x1, y1, x2, y2 = crop_box_for_size(w, h)
            cv2.imwrite(dst, img[y1:y2, x1:x2])
            total += 1
        if (fi + 1) % 200 == 0:
            print(f"  Cropping: {fi + 1}/{len(folders)} folders...")
    print(f"  Done! Cropped: {total} new, {skipped} already done, {corrupt} corrupt/skipped")


def load_depth_raw(path):
    if not path or not os.path.exists(path):
        return None
    size = os.path.getsize(path)
    expected = TOF_W * TOF_H * 2
    if size == expected + 8:
        raw = np.fromfile(path, dtype=np.uint16, offset=8)
    elif size == expected:
        raw = np.fromfile(path, dtype=np.uint16)
    else:
        return None
    return raw.reshape((TOF_H, TOF_W)) if raw.size == TOF_W * TOF_H else None


def check_pig_present(depth_raw_path):
    depth = load_depth_raw(depth_raw_path)
    if depth is None:
        return True, 0.0
    x1, y1, x2, y2 = CROP_TOF
    crop = depth[y1:y2, x1:x2].copy()
    crop[:, :BAR_MARGIN] = 0
    crop[:, -BAR_MARGIN:] = 0
    valid = crop[(crop > 200) & (crop < 5000)]
    if len(valid) == 0:
        return False, 0.0
    median = float(np.median(valid))
    return median < DEPTH_THRESHOLD, median


def scan_folder(data_dir):
    folders = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and FOLDER_RE.match(d))
    if not folders:
        return []
    records = {}
    for folder in folders:
        folder_path = os.path.join(data_dir, folder)
        for fname in os.listdir(folder_path):
            m = FILENAME_RE.match(fname)
            if not m:
                continue
            pig_id, modality, timestamp = int(m.group(1)), m.group(2), m.group(3)
            is_cropped = m.group(4) is not None
            key = (folder, pig_id, timestamp)
            if key not in records:
                records[key] = {"timestamp_folder": folder, "pig_id": pig_id, "pig_timestamp": timestamp}
            mod_key = "ir" if modality == "ir_vis" else "depth" if modality == "depth_vis" else modality
            records[key][f"{mod_key}{'_cropped' if is_cropped else ''}_{m.group(5)}"] = os.path.join(folder_path, fname)
    return sorted(records.values(), key=lambda r: (r["timestamp_folder"], r["pig_id"]))


def run_predictions(data_dir, device, model):
    frames = scan_folder(data_dir)
    if not frames:
        print("No frames found. Check your DATA_DIR path.")
        return []
    print(f"Running AI model on {len(frames)} frames...")
    results, skipped, skipped_corrupt = [], 0, 0
    with torch.no_grad():
        for i, frame in enumerate(frames):
            present, median_d = check_pig_present(frame.get("depth_raw", ""))
            if not present:
                skipped += 1
                continue
            path = ""
            for key in ("ir_cropped_jpg", "ir_jpg"):
                p = frame.get(key, "")
                if p and os.path.exists(p):
                    path = p
                    break
            if not path:
                skipped_corrupt += 1
                continue
            img = load_and_preprocess(path)
            if img is None:
                skipped_corrupt += 1
                continue
            x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)
            probs = torch.softmax(model(x), dim=1).cpu().numpy()[0]
            pred = int(np.argmax(probs))
            results.append({
                "timestamp_folder": frame["timestamp_folder"],
                "pig_id": frame["pig_id"],
                "pig_timestamp": frame["pig_timestamp"],
                "prediction": pred,
                "prediction_name": POSTURE3_CLASSES.get(pred, str(pred)),
                "confidence": round(float(probs[pred]), 4),
                "median_depth": round(median_d, 1),
                "image_path": path,
            })
            if (i + 1) % 500 == 0:
                pct = (i + 1) / len(frames) * 100
                print(f"  Progress: {i + 1}/{len(frames)} ({pct:.0f}%)")
    print(f"")
    print(f"  Predicted: {len(results)} frames")
    print(f"  Skipped (empty stall): {skipped}")
    print(f"  Skipped (corrupt/missing): {skipped_corrupt}")
    return results


def parse_ts(ts_str):
    try:
        return datetime.strptime(ts_str, "%Y%m%d-%H-%M-%S")
    except ValueError:
        return None


def _build_heatmap_grid(results):
    """Shared logic: build hours list and grid from results."""
    valid = [(r, t) for r in results for t in [parse_ts(r["pig_timestamp"])] if t]
    if not valid:
        return None, None, None
    min_t, max_t = min(t for _, t in valid), max(t for _, t in valid)
    hours = []
    t = min_t.replace(minute=0, second=0)
    while t <= max_t:
        hours.append(t)
        t += timedelta(hours=1)
    if not hours:
        return None, None, None
    return valid, hours, {"standing": 0, "sitting": 1, "lying": 2}


def generate_heatmap(results, out_path):
    """Full 20-pig heatmap."""
    for r in results:
        r["pig_id"] = r["pig_id"] % 20
    pig_ids = list(range(20))
    valid, hours, posture_map = _build_heatmap_grid(results)
    if valid is None:
        return
    votes = defaultdict(list)
    for r, t in valid:
        hi = min(int((t - hours[0]).total_seconds() / 3600), len(hours) - 1)
        votes[(r["pig_id"], hi)].append(posture_map.get(r["prediction_name"], -1))
    pig_idx = {pid: i for i, pid in enumerate(pig_ids)}
    grid = np.full((len(pig_ids), len(hours)), -1, dtype=float)
    for (pid, hi), postures in votes.items():
        if pid in pig_idx:
            grid[pig_idx[pid], hi] = max(set(postures), key=postures.count)
    cmap = ListedColormap(["#d4d4d4", "#22c55e", "#f97316", "#3b82f6"])
    norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], cmap.N)
    fig, ax = plt.subplots(figsize=(max(14, len(hours) * 0.3), 8))
    ax.pcolormesh(grid, cmap=cmap, norm=norm, edgecolors="white", linewidth=0.5)
    ax.set_yticks(np.arange(20) + 0.5)
    ax.set_yticklabels([f"pig {i}" for i in range(20)], fontsize=8)
    ax.set_ylim(0, 20)
    step = max(1, 6)
    ax.set_xticks(np.arange(0, len(hours), step) + 0.5)
    ax.set_xticklabels([hours[i].strftime("%m/%d %H:%M") for i in range(0, len(hours), step)], rotation=45, ha="right", fontsize=7)
    ax.set_xlim(0, len(hours))
    ax.set_xlabel("Time")
    ax.set_ylabel("Pig ID")
    ax.set_title("Posture Timeline Heatmap (hourly majority vote)")
    ax.legend(handles=[
        Patch(facecolor="#22c55e", label="Standing"),
        Patch(facecolor="#f97316", label="Sitting"),
        Patch(facecolor="#3b82f6", label="Lying"),
        Patch(facecolor="#d4d4d4", label="No data"),
    ], loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def generate_per_pig_heatmaps(results, out_dir):
    """One heatmap PNG per pig."""
    valid, hours, posture_map = _build_heatmap_grid(results)
    if valid is None:
        return
    cmap = ListedColormap(["#d4d4d4", "#22c55e", "#f97316", "#3b82f6"])
    norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], cmap.N)
    pig_ids = sorted(set(r["pig_id"] for r in results))
    pig_dir = os.path.join(out_dir, "per_pig")
    os.makedirs(pig_dir, exist_ok=True)
    for pid in pig_ids:
        pig_valid = [(r, t) for r, t in valid if r["pig_id"] == pid]
        if not pig_valid:
            continue
        votes = defaultdict(list)
        for r, t in pig_valid:
            hi = min(int((t - hours[0]).total_seconds() / 3600), len(hours) - 1)
            votes[hi].append(posture_map.get(r["prediction_name"], -1))
        grid = np.full((1, len(hours)), -1, dtype=float)
        for hi, postures in votes.items():
            grid[0, hi] = max(set(postures), key=postures.count)
        fig, ax = plt.subplots(figsize=(max(14, len(hours) * 0.15), 2))
        ax.pcolormesh(grid, cmap=cmap, norm=norm, edgecolors="white", linewidth=0.5)
        step = max(1, 6)
        ax.set_xticks(np.arange(0, len(hours), step) + 0.5)
        ax.set_xticklabels([hours[i].strftime("%m/%d %H:%M") for i in range(0, len(hours), step)], rotation=45, ha="right", fontsize=7)
        ax.set_xlim(0, len(hours))
        ax.set_yticks([0.5])
        ax.set_yticklabels([f"pig {pid}"], fontsize=9)
        ax.set_title(f"Pig {pid} — Posture Timeline", fontsize=11)
        ax.legend(handles=[
            Patch(facecolor="#22c55e", label="Standing"),
            Patch(facecolor="#f97316", label="Sitting"),
            Patch(facecolor="#3b82f6", label="Lying"),
            Patch(facecolor="#d4d4d4", label="No data"),
        ], loc="upper right", fontsize=7, ncol=4)
        plt.tight_layout()
        plt.savefig(os.path.join(pig_dir, f"pig{pid}_heatmap.png"), dpi=150)
        plt.close()
    print(f"  Per-pig heatmaps: {len(pig_ids)} pigs")


def generate_per_pig_csvs(results, out_dir):
    """One CSV per pig with predictions + summary stats."""
    pig_dir = os.path.join(out_dir, "per_pig")
    os.makedirs(pig_dir, exist_ok=True)
    pig_ids = sorted(set(r["pig_id"] for r in results))
    fields = ["timestamp_folder", "pig_timestamp", "prediction_name", "confidence", "median_depth"]
    for pid in pig_ids:
        pig_rows = [r for r in results if r["pig_id"] == pid]
        csv_path = os.path.join(pig_dir, f"pig{pid}_predictions.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(pig_rows)
    # Also make a summary sheet
    summary_rows = []
    for pid in pig_ids:
        pig_rows = [r for r in results if r["pig_id"] == pid]
        total = len(pig_rows)
        counts = Counter(r["prediction_name"] for r in pig_rows)
        summary_rows.append({
            "pig_id": pid,
            "total_frames": total,
            "standing": counts.get("standing", 0),
            "sitting": counts.get("sitting", 0),
            "lying": counts.get("lying", 0),
            "standing_pct": round(counts.get("standing", 0) / total * 100, 1) if total else 0,
            "sitting_pct": round(counts.get("sitting", 0) / total * 100, 1) if total else 0,
            "lying_pct": round(counts.get("lying", 0) / total * 100, 1) if total else 0,
            "avg_confidence": round(np.mean([r["confidence"] for r in pig_rows]), 4) if pig_rows else 0,
        })
    summary_path = os.path.join(out_dir, "pig_summary.csv")
    with open(summary_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"  Per-pig CSVs: {len(pig_ids)} files")
    print(f"  Pig summary: pig_summary.csv")


def generate_timestamp_summary(results, out_path):
    """Per-timestamp-folder breakdown."""
    folder_stats = defaultdict(lambda: {"standing": 0, "sitting": 0, "lying": 0, "total": 0, "conf_sum": 0.0})
    for r in results:
        s = folder_stats[r["timestamp_folder"]]
        s[r["prediction_name"]] += 1
        s["total"] += 1
        s["conf_sum"] += r["confidence"]
    rows = []
    for folder in sorted(folder_stats):
        s = folder_stats[folder]
        t = s["total"]
        rows.append({
            "timestamp_folder": folder,
            "total_frames": t,
            "standing": s["standing"],
            "sitting": s["sitting"],
            "lying": s["lying"],
            "standing_pct": round(s["standing"] / t * 100, 1),
            "sitting_pct": round(s["sitting"] / t * 100, 1),
            "lying_pct": round(s["lying"] / t * 100, 1),
            "avg_confidence": round(s["conf_sum"] / t, 4),
        })
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"  Timestamp summary: {len(rows)} folders")


print("Pipeline loaded! Ready to run.")

### Load the pre-trained model

The model is downloaded automatically — no setup needed.

In [ ]:
# Download the pre-trained model automatically from GitHub
MODEL_PATH = "/content/posture3_ir_best.pth"

if not os.path.exists(MODEL_PATH):
    print("Downloading pre-trained model...")
    !wget -q -O {MODEL_PATH} "https://github.com/AkbarDevop/paal-pipeline/releases/download/v1.0/posture3_ir_best.pth"

# If the download didn't work (repo is private), try Google Drive
if not os.path.exists(MODEL_PATH) or os.path.getsize(MODEL_PATH) < 1000:
    drive_paths = [
        "/content/drive/MyDrive/PAAL/posture3_ir_best.pth",
        "/content/drive/MyDrive/paal-pipeline/models/posture3_ir_best.pth",
        "/content/drive/MyDrive/posture3_ir_best.pth",
    ]
    found = False
    for p in drive_paths:
        if os.path.exists(p):
            MODEL_PATH = p
            found = True
            print(f"Found model on Drive: {p}")
            break
    if not found:
        print("")
        print("Model not found! Please do ONE of these:")
        print("")
        print("  Option A: Upload the model file manually")
        print("    - Click the folder icon on the left sidebar")
        print("    - Click the upload button")
        print("    - Upload 'posture3_ir_best.pth'")
        print("    - Then set: MODEL_PATH = '/content/posture3_ir_best.pth'")
        print("")
        print("  Option B: Put it on Google Drive")
        print("    - Upload posture3_ir_best.pth to your Google Drive")
        print("    - Set MODEL_PATH to the Drive path above")

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {'GPU' if device.type == 'cuda' else 'CPU (slower)'}")

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = SingleModalModel(
    in_channels=ckpt.get("in_channels", 3),
    num_classes=ckpt.get("num_classes", 3),
    pretrained=False,
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Model loaded! (MobileNetV2, 98.4% accuracy, 8.7MB)")

### Run classification

This processes all your images. Takes ~30 min for a full dataset.

In [ ]:
import time
start = time.time()

# Step 1: Crop images
print("="*50)
print("STEP 1/4: Cropping images...")
print("="*50)
crop_all_images(DATA_DIR)

# Step 2: Run AI model
print()
print("="*50)
print("STEP 2/4: Running posture classification...")
print("="*50)
results = run_predictions(DATA_DIR, device, model)

if results:
    # Normalize pig IDs for outputs
    for r in results:
        r["pig_id"] = r["pig_id"] % 20

    # Step 3: Save all CSVs
    print()
    print("="*50)
    print("STEP 3/4: Saving CSV files...")
    print("="*50)

    # Full predictions CSV
    csv_path = os.path.join(OUTPUT_DIR, "predictions.csv")
    fields = ["timestamp_folder", "pig_id", "pig_timestamp",
              "prediction", "prediction_name", "confidence",
              "median_depth", "image_path"]
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(results)
    print(f"  Full predictions: {len(results)} rows")

    # Per-pig CSVs + pig summary
    generate_per_pig_csvs(results, OUTPUT_DIR)

    # Timestamp summary
    ts_path = os.path.join(OUTPUT_DIR, "timestamp_summary.csv")
    generate_timestamp_summary(results, ts_path)

    # Step 4: Generate all heatmaps
    print()
    print("="*50)
    print("STEP 4/4: Generating heatmaps...")
    print("="*50)

    # Main heatmap (all pigs)
    heatmap_path = os.path.join(OUTPUT_DIR, "posture_heatmap.png")
    generate_heatmap(list(results), heatmap_path)
    print(f"  Main heatmap: posture_heatmap.png")

    # Per-pig heatmaps
    generate_per_pig_heatmaps(results, OUTPUT_DIR)

    # Summary
    elapsed = time.time() - start
    counts = Counter(r["prediction_name"] for r in results)
    total = len(results)
    print()
    print("="*50)
    print("  RESULTS")
    print("="*50)
    print(f"  Total frames classified: {total}")
    print(f"")
    for name in ["standing", "sitting", "lying"]:
        c = counts.get(name, 0)
        print(f"  {name:<10}: {c:>6} ({c/total*100:.1f}%)")
    print(f"")
    print(f"  Avg confidence: {np.mean([r['confidence'] for r in results]):.3f}")
    print(f"  Time taken: {elapsed/60:.1f} minutes")
    print(f"")
    print(f"  Output files:")
    print(f"    predictions.csv          — every frame prediction")
    print(f"    pig_summary.csv          — per-pig posture breakdown")
    print(f"    timestamp_summary.csv    — per-folder breakdown")
    print(f"    posture_heatmap.png      — all pigs timeline")
    print(f"    per_pig/pig0_heatmap.png — individual pig timelines")
    print(f"    per_pig/pig0_predictions.csv — individual pig data")
    print("="*50)
else:
    print("")
    print("No images found! Check your DATA_DIR path in Step 2.")

## Step 4: View Results

In [ ]:
# Show the main heatmap (all 20 pigs)
from IPython.display import Image, display

heatmap_file = os.path.join(OUTPUT_DIR, "posture_heatmap.png")
if os.path.exists(heatmap_file):
    print("ALL PIGS — Posture Heatmap")
    print("Green = Standing  |  Orange = Sitting  |  Blue = Lying  |  Gray = No data")
    print()
    display(Image(filename=heatmap_file))
else:
    print("Run Step 3 first!")

In [ ]:
# Show per-pig summary table
summary_file = os.path.join(OUTPUT_DIR, "pig_summary.csv")
if os.path.exists(summary_file):
    df = pd.read_csv(summary_file)
    print("PER-PIG SUMMARY")
    print()
    display(df)
else:
    print("Run Step 3 first!")

In [ ]:
# Show individual pig heatmaps (pick which pig to view)
PIG_TO_VIEW = 0  # Change this number to view a different pig (0-19)

pig_heatmap = os.path.join(OUTPUT_DIR, "per_pig", f"pig{PIG_TO_VIEW}_heatmap.png")
if os.path.exists(pig_heatmap):
    display(Image(filename=pig_heatmap))
else:
    print(f"No heatmap for pig {PIG_TO_VIEW}. Run Step 3 first or try a different pig ID.")

In [ ]:
# Preview predictions for a specific pig
PIG_TO_VIEW = 0  # Change this number

pig_csv = os.path.join(OUTPUT_DIR, "per_pig", f"pig{PIG_TO_VIEW}_predictions.csv")
if os.path.exists(pig_csv):
    df = pd.read_csv(pig_csv)
    print(f"Pig {PIG_TO_VIEW}: {len(df)} predictions")
    print()
    display(df.head(20))
else:
    print(f"No data for pig {PIG_TO_VIEW}.")

## Step 5: Download All Results

Downloads everything as one zip file — all CSVs, all heatmaps, everything.

In [ ]:
from google.colab import files

# Zip everything into one file
zip_path = "/content/paal_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

print("Downloading paal_results.zip...")
print()
print("Contents:")
print("  predictions.csv          — all predictions (every frame)")
print("  pig_summary.csv          — per-pig posture percentages")
print("  timestamp_summary.csv    — per-timestamp breakdown")
print("  posture_heatmap.png      — main heatmap (all pigs)")
print("  per_pig/                 — individual pig files:")
print("    pig0_heatmap.png         — pig 0 timeline")
print("    pig0_predictions.csv     — pig 0 predictions")
print("    pig1_heatmap.png         — pig 1 timeline")
print("    ...                      — (all 20 pigs)")
print()

files.download(f"{zip_path}.zip")